# 02 · Typed — ontology-guided extraction

cognify was run with a domain ontology (`ontology.ttl`), so the graph follows that vocabulary: ontology classes become `EntityType` nodes and matched entities are flagged `ontology_valid`.

> Run `build.py` first.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # examples/demos
from _common import config
config.require_openai_key(); config.quiet()          # quiet cognee's verbose logs
config.configure("cognee_typed")
import cognee
from cognee.modules.search.types import SearchType
from cognee.infrastructure.databases.graph import get_graph_engine

def _name(node):
    if isinstance(node, dict):
        return str(node.get("name") or (node.get("text") or "")[:40] or node.get("id") or "?")
    return str(node)[:40]

def render(results):
    if isinstance(results, (str, bytes)) or not isinstance(results, (list, tuple)):
        results = [results] if results else []
    for r in results[:4]:
        if isinstance(r, (tuple, list)) and len(r) == 3:
            src, edge, tgt = r
            rel = edge.get("relationship_name") if isinstance(edge, dict) else str(edge)
            print(f"   ({_name(src)}) -[{rel}]-> ({_name(tgt)})")
        elif isinstance(r, dict):
            print("   " + str(r.get("text") or r.get("name") or r)[:150])
        else:
            print("   " + str(r).strip().replace(chr(10), " ")[:550])
nodes, _ = await (await get_graph_engine()).get_graph_data()
entity_types = {p.get("name") for _, p in nodes if p.get("type") == "EntityType"}
aligned = [p.get("name") for _, p in nodes if p.get("ontology_valid") and p.get("type") == "Entity"]
print("EntityType nodes total:", len(entity_types))
print("entities aligned to the ontology (ontology_valid):", len(aligned))
print("aligned:", ", ".join(map(str, aligned[:12])))

EntityType nodes total: 261
entities aligned to the ontology (ontology_valid): 21
aligned: alabama, america_the_beautiful, academy_award_for_best_production_design, alaska, alain_connes, animalia, actrius, anarchism, asia, american_revolutionary_war, allan_dwan, alien_film_franchise


## Typed triplets (`INSIGHTS`) and a grounded answer

In [2]:
q = f"What is {aligned[0]} and what is it connected to?" if aligned else "What are the key entities?"
print("Q:", q, "\n")
render(await config.search(query_text=q, query_type=SearchType.INSIGHTS))
print()
render(await config.search(query_text=q, query_type=SearchType.GRAPH_COMPLETION))

Q: What is alabama and what is it connected to? 



   (# Alabama

Alabama () is a state in the ) -[contains]-> (alabama)
   (1819-12-14) -[statehood_date]-> (alabama)
   (alabama) -[is_a]-> (place)
   (alabama) -[is_a]-> (location)



   Alabama is a state in the Southeastern region of the United States, bordered by Tennessee to the north, Georgia to the east, Florida and the Gulf of Mexico to the south, and Mississippi to the west. It is known as the Yellowhammer State and its largest city is Huntsville. Alabama is historically significant for its role in the American Civil War, where it seceded from the United States.
